In [0]:
# Load Bronze tables
orders_bronze = spark.table("default.orders")
customers_bronze = spark.table("default.customers")
products_bronze = spark.table("default.products")

In [0]:
#Clean Orders
from pyspark.sql.functions import col, to_timestamp

orders_clean = (
    orders_bronze.dropDuplicates()                # Remove duplicates
                 .filter(col("customer_id").isNotNull())  # Remove rows with missing customer_id
                 .withColumn("timestamp", to_timestamp("timestamp")) # Convert timestamp to proper type
)

orders_clean.show()

+--------+-----------+----------+--------+-----+-------------------+
|order_id|customer_id|product_id|quantity|price|          timestamp|
+--------+-----------+----------+--------+-----+-------------------+
|       1|        101|      2001|       2| 15.5|2024-01-01 10:00:00|
|       2|        102|      2002|       1| 20.0|2024-01-01 11:00:00|
|       4|        101|      2001|       1| 15.5|2024-01-02 09:00:00|
|       5|        103|      2002|       3| 20.0|2024-01-02 10:00:00|
+--------+-----------+----------+--------+-----+-------------------+



In [0]:
#Enrich Orders (Join with Customers & Products)
orders_silver = (
    orders_clean
    .join(customers_bronze, "customer_id", "left")
    .join(products_bronze, "product_id", "left")
)

orders_silver.show()

+----------+-----------+--------+--------+-----+-------------------+-------------+-----------------+-------+------------+-----------+
|product_id|customer_id|order_id|quantity|price|          timestamp|customer_name|            email|country|product_name|   category|
+----------+-----------+--------+--------+-----+-------------------+-------------+-----------------+-------+------------+-----------+
|      2002|        102|       2|       1| 20.0|2024-01-01 11:00:00|          Bob|    bob@email.com|Germany|       Phone|Electronics|
|      2001|        101|       4|       1| 15.5|2024-01-02 09:00:00|        Alice|  alice@email.com| France|      Laptop|Electronics|
|      2002|        103|       5|       3| 20.0|2024-01-02 10:00:00|      Charlie|charlie@email.com|Belgium|       Phone|Electronics|
|      2001|        101|       1|       2| 15.5|2024-01-01 10:00:00|        Alice|  alice@email.com| France|      Laptop|Electronics|
+----------+-----------+--------+--------+-----+--------------

In [0]:
#Save Silver Table
orders_silver.write.format("delta").mode("overwrite").saveAsTable("default.orders_silver")

In [0]:
#Verify Silver Table
df = spark.table("default.orders_silver")
df.show()

+----------+-----------+--------+--------+-----+-------------------+-------------+-----------------+-------+------------+-----------+
|product_id|customer_id|order_id|quantity|price|          timestamp|customer_name|            email|country|product_name|   category|
+----------+-----------+--------+--------+-----+-------------------+-------------+-----------------+-------+------------+-----------+
|      2002|        102|       2|       1| 20.0|2024-01-01 11:00:00|          Bob|    bob@email.com|Germany|       Phone|Electronics|
|      2001|        101|       4|       1| 15.5|2024-01-02 09:00:00|        Alice|  alice@email.com| France|      Laptop|Electronics|
|      2002|        103|       5|       3| 20.0|2024-01-02 10:00:00|      Charlie|charlie@email.com|Belgium|       Phone|Electronics|
|      2001|        101|       1|       2| 15.5|2024-01-01 10:00:00|        Alice|  alice@email.com| France|      Laptop|Electronics|
+----------+-----------+--------+--------+-----+--------------